In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/small_bench/checkpoints/pre_cell_19.pickle

In [4]:
%%RecordEvent
%%time
### cell 19 ###

# Vectorized GPU datetime index conversion without CPU-bound to_datetime
months = births_by_date.index.get_level_values(0)
days   = births_by_date.index.get_level_values(1)
# Cumulative days before each month in 2020 (leap year)
month_offsets = pd.Series([0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335], dtype="int32")
# Compute day-of-year (0-based)
day_of_year = month_offsets.take(months - 1) + days - 1
# Days from 1970-01-01 to 2020-01-01 = 18262
epoch_days = 18262
# Nanoseconds per day
ns_per_day = 86400 * 10**9
# Build nanosecond timestamps and convert on GPU
ts_ns = (epoch_days + day_of_year) * ns_per_day
births_by_date.index = pd.to_datetime(ts_ns, unit="ns")
births_by_date.head()

CPU times: user 17.2 ms, sys: 8.53 ms, total: 25.8 ms
Wall time: 25.8 ms


,births
2020-01-01,4009.225
2020-01-02,4247.400
2020-01-03,4500.900
2020-01-04,4571.350
2020-01-05,4603.625


In [ ]:
%Checkpoint /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_small_rewrite/checkpoints/post_cell_19_try_1.pickle